In [1]:
import pandas as pd
import numpy as np

In [3]:
import pandas as pd

# -----------------------------
# 1. LOAD ALL 3 TABLES
# -----------------------------

orders = pd.read_csv(r"D:\Personal\Data Science\Data Science\End to End Project\SQL Dump\fact_orders.csv")
order_items = pd.read_csv(r"D:\Personal\Data Science\Data Science\End to End Project\SQL Dump\fact_order_items.csv")
products = pd.read_csv(r"D:\Personal\Data Science\Data Science\End to End Project\SQL Dump\dim_products.csv")

# -----------------------------
# 2. SELECT ONLY REQUIRED COLUMNS
# (avoid unnecessary data early)
# -----------------------------

orders = orders[['order_id', 'customer_id', 'order_date']]
order_items = order_items[['order_id', 'product_id']]
products = products[['product_id', 'category']]

# -----------------------------
# 3. CONVERT DATE COLUMN
# (VERY IMPORTANT for later rolling windows)
# -----------------------------

orders['order_date'] = pd.to_datetime(orders['order_date'])

# -----------------------------
# 4. JOIN: orders + order_items
# This connects WHO + WHAT
# -----------------------------

df = pd.merge(
    orders,
    order_items,
    on='order_id',
    how='inner'
)

# -----------------------------
# 5. JOIN: result + products
# This connects WHAT → CATEGORY
# -----------------------------

df = pd.merge(
    df,
    products,
    on='product_id',
    how='inner'
)

# -----------------------------
# 6. FINAL BASE TABLE
# Keep only what we need
# -----------------------------

base_df = df[['customer_id', 'order_date', 'category']]

# -----------------------------
# 7. REMOVE DUPLICATES
# (IMPORTANT: same user might buy multiple items in same category in same order)
# -----------------------------

base_df = base_df.drop_duplicates()

# -----------------------------
# 8. SORT (helps debugging & validation)
# -----------------------------

base_df = base_df.sort_values(['customer_id', 'order_date'])

# -----------------------------
# 9. RESET INDEX
# -----------------------------

base_df = base_df.reset_index(drop=True)

# -----------------------------
# DONE: base_df is ready
# -----------------------------

print(base_df.head())

   customer_id order_date        category
0            1 2024-05-18          Beauty
1            1 2024-05-22          Beauty
2            1 2024-05-22     Electronics
3            1 2024-06-27     Electronics
4            1 2024-09-11  Home & Kitchen


In [7]:
base_df.shape

(5251, 3)

In [8]:
base_df.isna().sum()

customer_id    0
order_date     0
category       0
dtype: int64

In [9]:
base_df.duplicated().sum()

np.int64(0)

In [10]:
base_df['category'].value_counts()

category
Electronics       1612
Home & Kitchen    1258
Beauty            1214
Fashion           1167
Name: count, dtype: int64

In [11]:
print(base_df['order_date'].min(), base_df['order_date'].max())

2024-01-01 00:00:00 2024-12-31 00:00:00


In [12]:
base_df

,customer_id,order_date,category
0,1,2024-05-18,Beauty
1,1,2024-05-22,Beauty
2,1,2024-05-22,Electronics
3,1,2024-06-27,Electronics
4,1,2024-09-11,Home & Kitchen
...,...,...,...
5246,1499,2024-09-21,Fashion
5247,1499,2024-09-21,Electronics
5248,1500,2024-12-13,Home & Kitchen
5249,1500,2024-12-13,Electronics


In [22]:
import pandas as pd

# -----------------------------
# 1. COPY BASE DATA
# -----------------------------

df = base_df.copy()

# -----------------------------
# 2. CREATE MONTH-END DATE
# (Align with reference dashboard)
# -----------------------------

df['Date'] = df['order_date'].dt.to_period('M').dt.to_timestamp('M')

# Now all dates look like:
# 2023-01-31, 2023-02-28, etc.

# -----------------------------
# 3. CREATE TIME SPINE
# (All unique months sorted)
# -----------------------------

all_dates = sorted(df['Date'].unique())

# -----------------------------
# 4. DEFINE CATEGORY → COLUMN MAP
# -----------------------------

category_map = {
    'Electronics': 'A',
    'Home & Kitchen': 'B',
    'Beauty': 'C',
    'Fashion': 'D'
}

df['cat_code'] = df['category'].map(category_map)

# -----------------------------
# 5. FUNCTION TO GET WINDOW FILTER
# -----------------------------

def get_window_data(data, current_date, window):
    
    if window == 'M':
        # Only that month
        start_date = current_date
    elif window == 'Q':
        # Last 3 months
        start_date = current_date - pd.DateOffset(months=2)
    elif window == 'Y':
        # Last 12 months
        start_date = current_date - pd.DateOffset(months=11)
    
    return data[
        (data['Date'] >= start_date) &
        (data['Date'] <= current_date)
    ]

# -----------------------------
# 6. BUILD USER MATRIX FOR EACH DATE + WINDOW
# -----------------------------

all_matrices = []

for current_date in all_dates:
    
    for window in ['M', 'Q', 'Y']:
        
        # -----------------------------
        # Filter data for this window
        # -----------------------------
        
        temp = get_window_data(df, current_date, window)
        
        # -----------------------------
        # Create user-category matrix
        # -----------------------------
        
        matrix = pd.crosstab(
            temp['customer_id'],
            temp['cat_code']
        )
        
        matrix.columns.name = None
        # -----------------------------
        # Ensure all 4 columns exist
        # -----------------------------
        
        for col in ['A','B','C','D']:
            if col not in matrix.columns:
                matrix[col] = 0
        
        matrix = matrix[['A','B','C','D']]
        
        # Convert counts → binary (important)
        matrix = (matrix > 0).astype(int)
        
        # -----------------------------
        # Add metadata
        # -----------------------------
        
        matrix['Date'] = current_date
        matrix['Window'] = window
        
        matrix = matrix.reset_index()
        
        # -----------------------------
        # Store result
        # -----------------------------
        
        all_matrices.append(matrix)

# -----------------------------
# 7. FINAL USER MATRIX DATASET
# -----------------------------

user_matrix_df = pd.concat(all_matrices, ignore_index=True)

print(user_matrix_df.head())

   customer_id  A  B  C  D       Date Window
0            2  0  0  1  0 2024-01-31      M
1           27  1  0  0  0 2024-01-31      M
2           80  1  0  0  0 2024-01-31      M
3          133  0  0  1  0 2024-01-31      M
4          141  1  0  0  0 2024-01-31      M


In [23]:
user_matrix_df.shape

(16640, 7)

In [26]:
# -----------------------------
# 1. COPY USER MATRIX
# -----------------------------

df = user_matrix_df.copy()

# -----------------------------
# 2. CALCULATE NUMBER OF PRODUCTS
# -----------------------------

df['num_products'] = df[['A','B','C','D']].sum(axis=1)

# -----------------------------
# 3. BUCKET INTO 1 / 2 / 3+
# -----------------------------

def bucket(x):
    if x == 1:
        return 1
    elif x == 2:
        return 2
    elif x >= 3:
        return '3+'
    else:
        return 0  # (optional: users with 0 products, usually ignore)

df['Num Products'] = df['num_products'].apply(bucket)

# -----------------------------
# 4. REMOVE USERS WITH 0 PRODUCTS
# (they shouldn't exist, but safe)
# -----------------------------

df = df[df['Num Products'] != 0]

# -----------------------------
# 5. AGGREGATE USERS
# -----------------------------

num_products_df = (
    df.groupby(['Date', 'Window', 'Num Products'])
    .agg(Users=('customer_id', 'nunique'))
    .reset_index()
)

# -----------------------------
# 6. SORT FOR READABILITY
# -----------------------------
# -----------------------------
# 7. ADD CURRENT & PREVIOUS DATE
# -----------------------------

current_date = user_matrix_df['Date'].max()
previous_date = current_date - pd.DateOffset(months=1)

num_products_df['Current Date'] = current_date
num_products_df['Previous Date'] = previous_date



num_products_df = num_products_df[
    ['Current Date', 'Date', 'Num Products', 'Previous Date', 'Window', 'Users']
]

# -----------------------------
# DONE
# -----------------------------

print(num_products_df.head())

  Current Date       Date Num Products Previous Date Window  Users
0   2024-12-31 2024-01-31            1    2024-11-30      M     18
1   2024-12-31 2024-01-31            2    2024-11-30      M     10
2   2024-12-31 2024-01-31           3+    2024-11-30      M      2
3   2024-12-31 2024-01-31            1    2024-11-30      Q     18
4   2024-12-31 2024-01-31            2    2024-11-30      Q     10


In [27]:
num_products_df.shape

(108, 6)

In [31]:
import itertools

# -----------------------------
# 1. COPY DATA
# -----------------------------

df = user_matrix_df.copy()

# -----------------------------
# 2. DEFINE CATEGORY LIST
# -----------------------------

categories = ['A', 'B', 'C', 'D']

# -----------------------------
# 3. GENERATE ALL COMBINATIONS (1,2,3)
# -----------------------------

all_combos = []

for r in [1, 2, 3]:
    combos = list(itertools.combinations(categories, r))
    all_combos.extend(combos)

# -----------------------------
# 4. NORMALIZE INTO 3 COLUMNS
# -----------------------------

def normalize_combo(combo):
    combo = list(combo)
    combo = combo + ['None'] * (3 - len(combo))  # fill remaining
    return combo[:3]

# -----------------------------
# 5. FORMAT PRODUCT NAMES
# -----------------------------

def format_product(x):
    if x == 'None':
        return 'None'
    else:
        return f'Product {x}'

# -----------------------------
# 6. MAIN LOOP
# -----------------------------

results = []

for (date, window), group in df.groupby(['Date', 'Window']):
    
    # -----------------------------
    # SPECIAL ROW → ALL USERS
    # -----------------------------
    
    total_users = group['customer_id'].nunique()
    
    results.append({
        'Date': date,
        'Join Type': 'AND',
        'Product 1': 'All',
        'Product 2': 'All',
        'Product 3': 'All',
        'Window': window,
        'Users': total_users
    })
    
    # -----------------------------
    # LOOP THROUGH EACH COMBINATION
    # -----------------------------
    
    for combo in all_combos:
        
        p1, p2, p3 = normalize_combo(combo)
        
        # -----------------------------
        # AND LOGIC (strict intersection)
        # -----------------------------
        
        cond_and = (group[list(combo)].sum(axis=1) == len(combo))
        users_and = group[cond_and]['customer_id'].nunique()
        
        results.append({
            'Date': date,
            'Join Type': 'AND',
            'Product 1': format_product(p1),
            'Product 2': format_product(p2),
            'Product 3': format_product(p3),
            'Window': window,
            'Users': users_and
        })
        
        # -----------------------------
        # OR LOGIC (union)
        # -----------------------------
        
        cond_or = (group[list(combo)].sum(axis=1) >= 1)
        users_or = group[cond_or]['customer_id'].nunique()
        
        results.append({
            'Date': date,
            'Join Type': 'OR',
            'Product 1': format_product(p1),
            'Product 2': format_product(p2),
            'Product 3': format_product(p3),
            'Window': window,
            'Users': users_or
        })

# -----------------------------
# 7. CREATE FINAL DATAFRAME
# -----------------------------

combination_df = pd.DataFrame(results)

# -----------------------------
# 8. ADD CURRENT & PREVIOUS DATE
# -----------------------------

current_date = df['Date'].max()
previous_date = current_date - pd.DateOffset(months=1)

combination_df['Current Date'] = current_date
combination_df['Previous Date'] = previous_date

# -----------------------------
# 9. FINAL COLUMN ORDER
# -----------------------------

combination_df = combination_df[
    ['Current Date', 'Date', 'Join Type', 'Previous Date',
     'Product 1', 'Product 2', 'Product 3', 'Window', 'Users']
]

# -----------------------------
# 10. SORT (OPTIONAL BUT RECOMMENDED)
# -----------------------------

combination_df = combination_df.sort_values(
    ['Date', 'Window', 'Join Type', 'Product 1', 'Product 2', 'Product 3']
).reset_index(drop=True)

# -----------------------------
# DONE
# -----------------------------

print(combination_df.head())

  Current Date       Date Join Type Previous Date  Product 1  Product 2  \
0   2024-12-31 2024-01-31       AND    2024-11-30        All        All   
1   2024-12-31 2024-01-31       AND    2024-11-30  Product A       None   
2   2024-12-31 2024-01-31       AND    2024-11-30  Product A  Product B   
3   2024-12-31 2024-01-31       AND    2024-11-30  Product A  Product B   
4   2024-12-31 2024-01-31       AND    2024-11-30  Product A  Product B   

   Product 3 Window  Users  
0        All      M     30  
1       None      M     20  
2       None      M      3  
3  Product C      M      1  
4  Product D      M      0  


In [33]:
columns = ['Product 1', 'Product 2', 'Product 3']

for col in columns:
    print(f"Unique values in {col}: {combination_df[col].unique()}")

Unique values in Product 1: ['All' 'Product A' 'Product B' 'Product C' 'Product D']
Unique values in Product 2: ['All' 'None' 'Product B' 'Product C' 'Product D']
Unique values in Product 3: ['All' 'None' 'Product C' 'Product D']


In [32]:
combination_df.shape

(1044, 9)

In [80]:
num_products_df.groupby(['Date','Window'])['Users'].sum()

Date        Window
2024-01-31  M           30
            Q           30
            Y           30
2024-02-29  M           58
            Q           80
            Y           80
2024-03-31  M          119
            Q          182
            Y          182
2024-04-30  M          146
            Q          271
            Y          282
2024-05-31  M          198
            Q          370
            Y          399
2024-06-30  M          225
            Q          463
            Y          544
2024-07-31  M          291
            Q          571
            Y          689
2024-08-31  M          316
            Q          663
            Y          841
2024-09-30  M          332
            Q          740
            Y          963
2024-10-31  M          377
            Q          807
            Y         1103
2024-11-30  M          370
            Q          870
            Y         1233
2024-12-31  M          452
            Q          960
            Y         1373
Name: Use

In [81]:
user_matrix_df.groupby(['Date','Window'])['customer_id'].nunique()

Date        Window
2024-01-31  M           30
            Q           30
            Y           30
2024-02-29  M           58
            Q           80
            Y           80
2024-03-31  M          119
            Q          182
            Y          182
2024-04-30  M          146
            Q          271
            Y          282
2024-05-31  M          198
            Q          370
            Y          399
2024-06-30  M          225
            Q          463
            Y          544
2024-07-31  M          291
            Q          571
            Y          689
2024-08-31  M          316
            Q          663
            Y          841
2024-09-30  M          332
            Q          740
            Y          963
2024-10-31  M          377
            Q          807
            Y         1103
2024-11-30  M          370
            Q          870
            Y         1233
2024-12-31  M          452
            Q          960
            Y         1373
Name: cus

In [36]:
num_products_df['Num Products'].unique()

array([1, 2, '3+'], dtype=object)

In [37]:
num_products_df[num_products_df['Users'] <= 0]

,Current Date,Date,Num Products,Previous Date,Window,Users


In [82]:
all_rows = combination_df[
    (combination_df['Product 1'] == 'All') &
    (combination_df['Join Type'] == 'AND')
]

all_rows[['Date','Window','Users']]

,Date,Window,Users
11,2024-01-31,M,30
38,2024-01-31,Q,30
65,2024-01-31,Y,30
92,2024-02-29,M,58
121,2024-02-29,Q,80
150,2024-02-29,Y,80
179,2024-03-31,M,119
208,2024-03-31,Q,182
237,2024-03-31,Y,182
266,2024-04-30,M,146


In [83]:
num_products_df.groupby(['Date','Window'])['Users'].sum()

Date        Window
2024-01-31  M           30
            Q           30
            Y           30
2024-02-29  M           58
            Q           80
            Y           80
2024-03-31  M          119
            Q          182
            Y          182
2024-04-30  M          146
            Q          271
            Y          282
2024-05-31  M          198
            Q          370
            Y          399
2024-06-30  M          225
            Q          463
            Y          544
2024-07-31  M          291
            Q          571
            Y          689
2024-08-31  M          316
            Q          663
            Y          841
2024-09-30  M          332
            Q          740
            Y          963
2024-10-31  M          377
            Q          807
            Y         1103
2024-11-30  M          370
            Q          870
            Y         1233
2024-12-31  M          452
            Q          960
            Y         1373
Name: Use

In [84]:
combination_df[
    (combination_df['Product 1'] == 'Product A') &
    (combination_df['Product 2'] == 'Product B') &
    (combination_df['Product 3'] == 'None')
]

,Current Date,Date,Join Type,Previous Date,Product 1,Product 2,Product 3,Window,Users


In [42]:
single_A = combination_df[
    (combination_df['Product 1'] == 'Product A') &
    (combination_df['Product 2'] == 'None') &
    (combination_df['Product 3'] == 'None') &
    (combination_df['Join Type'] == 'AND')
]

single_A

,Current Date,Date,Join Type,Previous Date,Product 1,Product 2,Product 3,Window,Users
1,2024-12-31,2024-01-31,AND,2024-11-30,Product A,None,None,M,20
30,2024-12-31,2024-01-31,AND,2024-11-30,Product A,None,None,Q,20
59,2024-12-31,2024-01-31,AND,2024-11-30,Product A,None,None,Y,20
88,2024-12-31,2024-02-29,AND,2024-11-30,Product A,None,None,M,24
117,2024-12-31,2024-02-29,AND,2024-11-30,Product A,None,None,Q,43
146,2024-12-31,2024-02-29,AND,2024-11-30,Product A,None,None,Y,43
175,2024-12-31,2024-03-31,AND,2024-11-30,Product A,None,None,M,50
204,2024-12-31,2024-03-31,AND,2024-11-30,Product A,None,None,Q,87
233,2024-12-31,2024-03-31,AND,2024-11-30,Product A,None,None,Y,87
262,2024-12-31,2024-04-30,AND,2024-11-30,Product A,None,None,M,73


In [52]:
check_A_correct = user_matrix_df[
    user_matrix_df['A'] == 1
].groupby(['Date','Window'])['customer_id'].nunique()
check_A_correct

Date        Window
2024-01-31  M           20
            Q           20
            Y           20
2024-02-29  M           24
            Q           43
            Y           43
2024-03-31  M           50
            Q           87
            Y           87
2024-04-30  M           73
            Q          140
            Y          153
2024-05-31  M           91
            Q          195
            Y          223
2024-06-30  M          122
            Q          262
            Y          321
2024-07-31  M          155
            Q          332
            Y          438
2024-08-31  M          163
            Q          398
            Y          555
2024-09-30  M          165
            Q          433
            Y          653
2024-10-31  M          215
            Q          491
            Y          788
2024-11-30  M          198
            Q          521
            Y          908
2024-12-31  M          244
            Q          592
            Y         1046
Name: cus

In [44]:
combination_df.groupby(['Date','Window']).size()

Date        Window
2024-01-31  M         29
            Q         29
            Y         29
2024-02-29  M         29
            Q         29
            Y         29
2024-03-31  M         29
            Q         29
            Y         29
2024-04-30  M         29
            Q         29
            Y         29
2024-05-31  M         29
            Q         29
            Y         29
2024-06-30  M         29
            Q         29
            Y         29
2024-07-31  M         29
            Q         29
            Y         29
2024-08-31  M         29
            Q         29
            Y         29
2024-09-30  M         29
            Q         29
            Y         29
2024-10-31  M         29
            Q         29
            Y         29
2024-11-30  M         29
            Q         29
            Y         29
2024-12-31  M         29
            Q         29
            Y         29
dtype: int64

In [45]:
combination_df[
    combination_df['Product 1'].str.contains('None') &
    (combination_df['Product 1'] != 'None')
]

,Current Date,Date,Join Type,Previous Date,Product 1,Product 2,Product 3,Window,Users


In [46]:
combination_df[['Product 1','Product 2','Product 3']].stack().unique()

array(['All', 'Product A', 'None', 'Product B', 'Product C', 'Product D'],
      dtype=object)

In [47]:
sample = '2024-01-31'

In [48]:
total_users = user_matrix_df[
    (user_matrix_df['Date'] == sample) &
    (user_matrix_df['Window'] == 'M')
]['customer_id'].nunique()

print(f"Total users in {sample} (M): {total_users}")

Total users in 2024-01-31 (M): 30


In [49]:
all_users = combination_df[
    (combination_df['Date'] == sample) &
    (combination_df['Window'] == 'M') &
    (combination_df['Product 1'] == 'All')
]['Users'].values[0]

print(f"Total users in {sample} (M) from combination_df: {all_users}")

Total users in 2024-01-31 (M) from combination_df: 30


In [53]:
# Map back to real names:

label_map = {
    'Product A': 'Electronics',
    'Product B': 'Home & Kitchen',
    'Product C': 'Beauty',
    'Product D': 'Fashion',
    'None': 'None',
    'All': 'All'
}

for col in ['Product 1','Product 2','Product 3']:
    combination_df[col] = combination_df[col].map(label_map)

In [54]:
# Ensure Sorting Stability

combination_df = combination_df.sort_values(
    ['Date','Window','Join Type','Product 1','Product 2','Product 3']
).reset_index(drop=True)

num_products_df = num_products_df.sort_values(
    ['Date','Window','Num Products']
).reset_index(drop=True)

In [55]:
combination_df.to_csv("combinations_final.csv", index=False)
num_products_df.to_csv("num_products_final.csv", index=False)

In [56]:
user_matrix_df[
    user_matrix_df['A'] == 1
]['customer_id'].nunique()

1046

In [65]:
import itertools

# -----------------------------
# 1. COPY DATA
# -----------------------------

df = user_matrix_df.copy()

# -----------------------------
# 2. DEFINE CATEGORY LIST
# -----------------------------

categories = ['A', 'B', 'C', 'D']

# -----------------------------
# 3. GENERATE ALL COMBINATIONS (1,2,3)
# -----------------------------

all_combos = []

for r in [1, 2, 3]:
    combos = list(itertools.combinations(categories, r))
    all_combos.extend(combos)

# -----------------------------
# 4. NORMALIZE INTO 3 COLUMNS
# -----------------------------

def normalize_combo(combo):
    combo = list(combo)
    combo = combo + ['None'] * (3 - len(combo))  # fill remaining
    return combo[:3]

# -----------------------------
# 5. FORMAT PRODUCT NAMES
# -----------------------------

def format_product(x):
    if x == 'None':
        return 'None'
    elif x == 'All':
        return 'All'
    else:
        return f'Product {x}'

# -----------------------------
# 6. MAIN LOOP
# -----------------------------
# -----------------------------
# 6. MAIN LOOP (UPDATED WITH ALL EXPANSION)
# -----------------------------
results = []

for (date, window), group in df.groupby(['Date', 'Window']):
    
    # -----------------------------
    # LOOP THROUGH EACH COMBINATION
    # -----------------------------
    
    for combo in all_combos:
        
        p1, p2, p3 = normalize_combo(combo)
        
        # -----------------------------
        # AND LOGIC
        # -----------------------------
        
        cond_and = (group[list(combo)].sum(axis=1) == len(combo))
        users_and = group[cond_and]['customer_id'].nunique()
        
        # -----------------------------
        # OR LOGIC
        # -----------------------------
        
        cond_or = (group[list(combo)].sum(axis=1) >= 1)
        users_or = group[cond_or]['customer_id'].nunique()
        
        # -----------------------------
        # GENERATE ALL VARIANTS (None → All)
        # -----------------------------
        
        base_products = [p1, p2, p3]
        
        variants = set()
        
        # create all combinations of replacing with "All"
        for mask in itertools.product([True, False], repeat=3):
            variant = []
            for i in range(3):
                if mask[i]:
                    variant.append('All')
                else:
                    variant.append(base_products[i])
            variants.add(tuple(variant))
        
        # -----------------------------
        # APPEND ALL VARIANTS
        # -----------------------------
        
        for v1, v2, v3 in variants:
            
            results.append({
                'Date': date,
                'Join Type': 'AND',
                'Product 1': format_product(v1),
                'Product 2': format_product(v2),
                'Product 3': format_product(v3),
                'Window': window,
                'Users': users_and
            })
            
            results.append({
                'Date': date,
                'Join Type': 'OR',
                'Product 1': format_product(v1),
                'Product 2': format_product(v2),
                'Product 3': format_product(v3),
                'Window': window,
                'Users': users_or
            })
# -----------------------------
# 7. CREATE FINAL DATAFRAME
# -----------------------------

combination_df = pd.DataFrame(results)

# -----------------------------
# 8. ADD CURRENT & PREVIOUS DATE
# -----------------------------

current_date = df['Date'].max()
previous_date = current_date - pd.DateOffset(months=1)

combination_df['Current Date'] = current_date
combination_df['Previous Date'] = previous_date

# -----------------------------
# 9. FINAL COLUMN ORDER
# -----------------------------

combination_df = combination_df[
    ['Current Date', 'Date', 'Join Type', 'Previous Date',
     'Product 1', 'Product 2', 'Product 3', 'Window', 'Users']
]

# -----------------------------
# 10. SORT (OPTIONAL BUT RECOMMENDED)
# -----------------------------

combination_df_1 = combination_df.sort_values(
    ['Date', 'Window', 'Join Type', 'Product 1', 'Product 2', 'Product 3']
).reset_index(drop=True)

# -----------------------------
# DONE
# -----------------------------

print(combination_df.head())

  Current Date       Date Join Type Previous Date  Product 1 Product 2  \
0   2024-12-31 2024-01-31       AND    2024-11-30  Product A      None   
1   2024-12-31 2024-01-31        OR    2024-11-30  Product A      None   
2   2024-12-31 2024-01-31       AND    2024-11-30  Product A       All   
3   2024-12-31 2024-01-31        OR    2024-11-30  Product A       All   
4   2024-12-31 2024-01-31       AND    2024-11-30        All      None   

  Product 3 Window  Users  
0       All      M     20  
1       All      M     20  
2       All      M     20  
3       All      M     20  
4       All      M     20  


In [67]:
combination_df_1[
    (combination_df_1['Product 2'] == 'All')
]

,Current Date,Date,Join Type,Previous Date,Product 1,Product 2,Product 3,Window,Users
0,2024-12-31,2024-01-31,AND,2024-11-30,All,All,All,M,20
1,2024-12-31,2024-01-31,AND,2024-11-30,All,All,All,M,6
2,2024-12-31,2024-01-31,AND,2024-11-30,All,All,All,M,9
3,2024-12-31,2024-01-31,AND,2024-11-30,All,All,All,M,9
4,2024-12-31,2024-01-31,AND,2024-11-30,All,All,All,M,3
...,...,...,...,...,...,...,...,...,...
8053,2024-12-31,2024-12-31,OR,2024-11-30,Product C,All,All,Y,1191
8054,2024-12-31,2024-12-31,OR,2024-11-30,Product C,All,None,Y,910
8055,2024-12-31,2024-12-31,OR,2024-11-30,Product C,All,None,Y,1191
8060,2024-12-31,2024-12-31,OR,2024-11-30,Product D,All,All,Y,876


In [66]:
print(combination_df.shape)
print(combination_df_1.shape)

(8064, 9)
(8064, 9)


In [68]:
combination_df_1.to_csv("combinations_final_1.csv", index=False)

In [73]:

# -----------------------------
# INPUT: user_matrix_df
# Columns: customer_id, Date, Window, A, B, C, D
# -----------------------------

product_cols = ['A', 'B', 'C', 'D']

# -----------------------------
# FORMATTERS
# -----------------------------
def format_product(x):
    if x == 'None':
        return 'None'
    elif x == 'All':
        return 'All'
    else:
        return f'Product {x}'

def pad_to_three(seq):
    seq = list(seq)
    while len(seq) < 3:
        seq.append('None')
    return seq[:3]

def generate_hierarchy(p1, p2, p3):
    # Right → Left aggregation ONLY
    return [
        (p1, p2, p3),
        (p1, p2, 'All'),
        (p1, 'All', 'All'),
        ('All', 'All', 'All')
    ]

# -----------------------------
# MAIN LOGIC
# -----------------------------
results = []

for (date, window), group in user_matrix_df.groupby(['Date', 'Window']):
    
    for _, row in group.iterrows():
        
        # -----------------------------
        # Step 1: Get products used by user
        # -----------------------------
        user_products = [p for p in product_cols if row[p] == 1]
        
        if len(user_products) == 0:
            continue
        
        # -----------------------------
        # Step 2: Generate permutations (ORDERED)
        # -----------------------------
        max_len = min(len(user_products), 3)
        
        # Sort products (consistent ordering)
        user_products_sorted = sorted(user_products)

        # Take only first 3
        p1, p2, p3 = pad_to_three(user_products_sorted[:3])

        # Only ONE base combination
        base_combos = [(p1, p2, p3)]
        
        # -----------------------------
        # Step 3: Process each permutation
        # -----------------------------
        for perm in perms:
            
            p1, p2, p3 = pad_to_three(perm)
            
            # -----------------------------
            # Step 4: Hierarchy
            # -----------------------------
            variants = generate_hierarchy(p1, p2, p3)
            
            for v1, v2, v3 in variants:
                
                results.append({
                    'customer_id': row['customer_id'],
                    'Date': date,
                    'Window': window,
                    'Product 1': format_product(v1),
                    'Product 2': format_product(v2),
                    'Product 3': format_product(v3)
                })

# -----------------------------
# STEP 5: CREATE DF
# -----------------------------
temp_df = pd.DataFrame(results)

# -----------------------------
# STEP 6: AGGREGATE USERS
# -----------------------------
combination_df = (
    temp_df
    .groupby(['Date', 'Window', 'Product 1', 'Product 2', 'Product 3'])
    .agg(Users=('customer_id', 'nunique'))
    .reset_index()
)

# -----------------------------
# STEP 7: ADD JOIN TYPE (AND / OR)
# -----------------------------
# AND = exact combination match
# OR  = any of selected products (derived later in Tableau)

combination_df['Join Type'] = 'AND'

# Duplicate for OR (same base counts; Tableau handles interpretation)
or_df = combination_df.copy()
or_df['Join Type'] = 'OR'

combination_df = pd.concat([combination_df, or_df], ignore_index=True)

# -----------------------------
# STEP 8: ADD CURRENT / PREVIOUS DATE
# -----------------------------
current_date = user_matrix_df['Date'].max()
previous_date = current_date - pd.DateOffset(months=1)

combination_df['Current Date'] = current_date
combination_df['Previous Date'] = previous_date

# -----------------------------
# STEP 9: FINAL ORDER
# -----------------------------
combination_df = combination_df[
    [
        'Current Date',
        'Date',
        'Join Type',
        'Previous Date',
        'Product 1',
        'Product 2',
        'Product 3',
        'Window',
        'Users'
    ]
]

# -----------------------------
# VALIDATION
# -----------------------------
print("Shape:", combination_df.shape)
print("\nProduct 1 All count:", (combination_df['Product 1'] == 'All').sum())
print("Product 2 All count:", (combination_df['Product 2'] == 'All').sum())
print("Product 3 All count:", (combination_df['Product 3'] == 'All').sum())

print("\nSample:")
print(combination_df.head(10))

combination_df.to_csv('combinations_final_2.csv', index = False)



Shape: (2016, 9)

Product 1 All count: 72
Product 2 All count: 288
Product 3 All count: 936

Sample:
  Current Date       Date Join Type Previous Date  Product 1  Product 2  \
0   2024-12-31 2024-01-31       AND    2024-11-30        All        All   
1   2024-12-31 2024-01-31       AND    2024-11-30  Product A        All   
2   2024-12-31 2024-01-31       AND    2024-11-30  Product A       None   
3   2024-12-31 2024-01-31       AND    2024-11-30  Product A       None   
4   2024-12-31 2024-01-31       AND    2024-11-30  Product A  Product B   
5   2024-12-31 2024-01-31       AND    2024-11-30  Product A  Product B   
6   2024-12-31 2024-01-31       AND    2024-11-30  Product A  Product B   
7   2024-12-31 2024-01-31       AND    2024-11-30  Product A  Product D   
8   2024-12-31 2024-01-31       AND    2024-11-30  Product A  Product D   
9   2024-12-31 2024-01-31       AND    2024-11-30  Product A  Product D   

   Product 3 Window  Users  
0        All      M     30  
1        All   

In [70]:
combination_df_1.to_csv("combinations_final_n.csv", index=False)

In [74]:


# -----------------------------
# INPUT: user_matrix_df
# Columns: customer_id, Date, Window, A, B, C, D
# -----------------------------

product_cols = ['A', 'B', 'C', 'D']

# -----------------------------
# FORMATTERS
# -----------------------------
def format_product(x):
    if x == 'None':
        return 'None'
    elif x == 'All':
        return 'All'
    else:
        return f'Product {x}'

def pad_to_three(seq):
    seq = list(seq)
    while len(seq) < 3:
        seq.append('None')
    return seq[:3]

# -----------------------------
# HIERARCHY (RIGHT → LEFT ONLY)
# -----------------------------
def generate_hierarchy(p1, p2, p3):
    return [
        (p1, p2, p3),          # full
        (p1, p2, 'All'),       # drop product 3
        (p1, 'All', 'All'),    # drop product 2 & 3
        ('All', 'All', 'All')  # total
    ]

# -----------------------------
# MAIN LOGIC
# -----------------------------
results = []

for (date, window), group in user_matrix_df.groupby(['Date', 'Window']):
    
    for _, row in group.iterrows():
        
        # -----------------------------
        # Step 1: Get products used
        # -----------------------------
        user_products = [p for p in product_cols if row[p] == 1]
        
        if len(user_products) == 0:
            continue
        
        # -----------------------------
        # Step 2: SORT (canonical order)
        # -----------------------------
        user_products_sorted = sorted(user_products)
        
        # -----------------------------
        # Step 3: Take only first 3
        # -----------------------------
        p1, p2, p3 = pad_to_three(user_products_sorted[:3])
        
        # -----------------------------
        # Step 4: Generate hierarchy
        # -----------------------------
        variants = generate_hierarchy(p1, p2, p3)
        
        for v1, v2, v3 in variants:
            
            results.append({
                'customer_id': row['customer_id'],
                'Date': date,
                'Window': window,
                'Product 1': format_product(v1),
                'Product 2': format_product(v2),
                'Product 3': format_product(v3)
            })

# -----------------------------
# CREATE TEMP DF
# -----------------------------
temp_df = pd.DataFrame(results)

# -----------------------------
# AGGREGATE USERS (DEDUP)
# -----------------------------
combination_df = (
    temp_df
    .groupby(['Date', 'Window', 'Product 1', 'Product 2', 'Product 3'])
    .agg(Users=('customer_id', 'nunique'))
    .reset_index()
)

# -----------------------------
# ADD JOIN TYPES
# -----------------------------
combination_df['Join Type'] = 'AND'

or_df = combination_df.copy()
or_df['Join Type'] = 'OR'

combination_df = pd.concat([combination_df, or_df], ignore_index=True)

# -----------------------------
# ADD CURRENT / PREVIOUS DATE
# -----------------------------
current_date = user_matrix_df['Date'].max()
previous_date = current_date - pd.DateOffset(months=1)

combination_df['Current Date'] = current_date
combination_df['Previous Date'] = previous_date

# -----------------------------
# FINAL COLUMN ORDER
# -----------------------------
combination_df = combination_df[
    [
        'Current Date',
        'Date',
        'Join Type',
        'Previous Date',
        'Product 1',
        'Product 2',
        'Product 3',
        'Window',
        'Users'
    ]
]

# -----------------------------
# VALIDATION
# -----------------------------
print("Shape:", combination_df.shape)

print("\nProduct 1 All count:", (combination_df['Product 1'] == 'All').sum())
print("Product 2 All count:", (combination_df['Product 2'] == 'All').sum())
print("Product 3 All count:", (combination_df['Product 3'] == 'All').sum())

print("\nProduct 2 Distribution:")
print(combination_df['Product 2'].value_counts())

print("\nSample:")
print(combination_df.head(10))


Shape: (2074, 9)

Product 1 All count: 72
Product 2 All count: 360
Product 3 All count: 1080

Product 2 Distribution:
Product 2
None         576
Product D    432
Product C    424
All          360
Product B    282
Name: count, dtype: int64

Sample:
  Current Date       Date Join Type Previous Date  Product 1  Product 2  \
0   2024-12-31 2024-01-31       AND    2024-11-30        All        All   
1   2024-12-31 2024-01-31       AND    2024-11-30  Product A        All   
2   2024-12-31 2024-01-31       AND    2024-11-30  Product A       None   
3   2024-12-31 2024-01-31       AND    2024-11-30  Product A       None   
4   2024-12-31 2024-01-31       AND    2024-11-30  Product A  Product B   
5   2024-12-31 2024-01-31       AND    2024-11-30  Product A  Product B   
6   2024-12-31 2024-01-31       AND    2024-11-30  Product A  Product B   
7   2024-12-31 2024-01-31       AND    2024-11-30  Product A  Product C   
8   2024-12-31 2024-01-31       AND    2024-11-30  Product A  Product C   
9 

In [76]:
combination_df.to_csv('combinations_final_main.csv',index = False)

In [ ]:
# finall used code to make the combinations table

# -----------------------------
# INPUT: user_matrix_df
# Columns:
# customer_id, Date, Window, A, B, C, D
# -----------------------------

product_cols = ['A', 'B', 'C', 'D']

# -----------------------------
# CATEGORY MAPPING (FINAL STEP)
# -----------------------------
category_map = {
    'A': 'Electronics',
    'B': 'Home & Kitchen',
    'C': 'Beauty',
    'D': 'Fashion',
    'None': 'None',
    'All': 'All'
}

# -----------------------------
# PAD TO 3 SLOTS
# -----------------------------
def pad_to_three(seq):
    seq = list(seq)
    while len(seq) < 3:
        seq.append('None')
    return seq[:3]

# -----------------------------
# HIERARCHY (RIGHT → LEFT ONLY)
# -----------------------------
def generate_hierarchy(p1, p2, p3):
    return [
        (p1, p2, p3),
        (p1, p2, 'All'),
        (p1, 'All', 'All'),
        ('All', 'All', 'All')
    ]

# -----------------------------
# MAIN LOGIC
# -----------------------------
results = []

for (date, window), group in user_matrix_df.groupby(['Date', 'Window']):
    
    for _, row in group.iterrows():
        
        # Step 1: Get products used
        user_products = [p for p in product_cols if row[p] == 1]
        
        if len(user_products) == 0:
            continue
        
        # Step 2: Canonical ordering
        user_products_sorted = sorted(user_products)
        
        # Step 3: Take max 3
        p1, p2, p3 = pad_to_three(user_products_sorted[:3])
        
        # Step 4: Hierarchy
        variants = generate_hierarchy(p1, p2, p3)
        
        for v1, v2, v3 in variants:
            results.append({
                'customer_id': row['customer_id'],
                'Date': date,
                'Window': window,
                'Product 1': v1,
                'Product 2': v2,
                'Product 3': v3
            })

# -----------------------------
# CREATE TEMP DF
# -----------------------------
temp_df = pd.DataFrame(results)

# -----------------------------
# AGGREGATE USERS
# -----------------------------
combination_df = (
    temp_df
    .groupby(['Date', 'Window', 'Product 1', 'Product 2', 'Product 3'])
    .agg(Users=('customer_id', 'nunique'))
    .reset_index()
)

# -----------------------------
# ADD JOIN TYPES
# -----------------------------
combination_df['Join Type'] = 'AND'

or_df = combination_df.copy()
or_df['Join Type'] = 'OR'

combination_df = pd.concat([combination_df, or_df], ignore_index=True)

# -----------------------------
# ADD CURRENT / PREVIOUS DATE
# -----------------------------
current_date = user_matrix_df['Date'].max()
previous_date = current_date - pd.DateOffset(months=1)

combination_df['Current Date'] = current_date
combination_df['Previous Date'] = previous_date

# -----------------------------
# APPLY CATEGORY MAPPING (FINAL STEP)
# -----------------------------
for col in ['Product 1', 'Product 2', 'Product 3']:
    combination_df[col] = combination_df[col].map(category_map)

# -----------------------------
# FINAL COLUMN ORDER
# -----------------------------
combination_df = combination_df[
    [
        'Current Date',
        'Date',
        'Join Type',
        'Previous Date',
        'Product 1',
        'Product 2',
        'Product 3',
        'Window',
        'Users'
    ]
]

# -----------------------------
# VALIDATION
# -----------------------------
print("Shape:", combination_df.shape)

print("\nProduct 1 All count:", (combination_df['Product 1'] == 'All').sum())
print("Product 2 All count:", (combination_df['Product 2'] == 'All').sum())
print("Product 3 All count:", (combination_df['Product 3'] == 'All').sum())

print("\nProduct 2 Distribution:")
print(combination_df['Product 2'].value_counts())

print("\nSample:")
print(combination_df.head(10))



Shape: (2074, 9)

Product 1 All count: 72
Product 2 All count: 360
Product 3 All count: 1080

Product 2 Distribution:
Product 2
None              576
Fashion           432
Beauty            424
All               360
Home & Kitchen    282
Name: count, dtype: int64

Sample:
  Current Date       Date Join Type Previous Date    Product 1  \
0   2024-12-31 2024-01-31       AND    2024-11-30  Electronics   
1   2024-12-31 2024-01-31       AND    2024-11-30  Electronics   
2   2024-12-31 2024-01-31       AND    2024-11-30  Electronics   
3   2024-12-31 2024-01-31       AND    2024-11-30  Electronics   
4   2024-12-31 2024-01-31       AND    2024-11-30  Electronics   
5   2024-12-31 2024-01-31       AND    2024-11-30  Electronics   
6   2024-12-31 2024-01-31       AND    2024-11-30  Electronics   
7   2024-12-31 2024-01-31       AND    2024-11-30  Electronics   
8   2024-12-31 2024-01-31       AND    2024-11-30  Electronics   
9   2024-12-31 2024-01-31       AND    2024-11-30  Electronics   



In [79]:
combination_df.to_csv('combinations_final_mainf.csv',index = False)

In [95]:

product_cols = ['A','B','C','D']

category_map = {
    'A': 'Electronics',
    'B': 'Home & Kitchen',
    'C': 'Beauty',
    'D': 'Fashion',
    'None': 'None',
    'All': 'All'
}

def pad(combo):
    combo = list(combo)
    while len(combo) < 3:
        combo.append('None')
    return combo

# rotate function
def rotations(lst):
    return [lst[i:] + lst[:i] for i in range(len(lst))]

results = []

for (date, window), group in user_matrix_df.groupby(['Date','Window']):
    
    for r in range(1,4):
        for combo in itertools.combinations(product_cols, r):
            
            padded = pad(combo)
            
            # generate rotations
            rotated_versions = rotations(padded)
            
            for p1,p2,p3 in rotated_versions:

    # ❗ FIX: enforce strict None ordering
                if p2 == 'None' and p3 != 'None':
                    continue

    # existing rule
                if p1 == 'None':
                    continue
                
                active = [x for x in [p1,p2,p3] if x != 'None']
                
                # AND
                users_and = group[
                    (group[active].sum(axis=1) == len(active))
                ]['customer_id'].nunique()
                
                # OR
                users_or = group[
                    (group[active].sum(axis=1) >= 1)
                ]['customer_id'].nunique()
                
                results.append({
                    'Date': date,
                    'Window': window,
                    'Join Type': 'AND',
                    'Product 1': p1,
                    'Product 2': p2,
                    'Product 3': p3,
                    'Users': users_and
                })
                
                results.append({
                    'Date': date,
                    'Window': window,
                    'Join Type': 'OR',
                    'Product 1': p1,
                    'Product 2': p2,
                    'Product 3': p3,
                    'Users': users_or
                })
    
    # total
    total = group['customer_id'].nunique()
    
    results.append({
        'Date': date,
        'Window': window,
        'Join Type': 'AND',
        'Product 1': 'All',
        'Product 2': 'All',
        'Product 3': 'All',
        'Users': total
    })
    
    results.append({
        'Date': date,
        'Window': window,
        'Join Type': 'OR',
        'Product 1': 'All',
        'Product 2': 'All',
        'Product 3': 'All',
        'Users': total
    })

combination_df = pd.DataFrame(results)

# dates
current_date = user_matrix_df['Date'].max()
previous_date = current_date - pd.DateOffset(months=1)

combination_df['Current Date'] = current_date
combination_df['Previous Date'] = previous_date

# map names
for col in ['Product 1','Product 2','Product 3']:
    combination_df[col] = combination_df[col].map(category_map)

combination_df = combination_df[
    ['Current Date','Date','Join Type','Previous Date',
     'Product 1','Product 2','Product 3','Window','Users']
]

print("Shape:", combination_df.shape)
print("Product 2 None:", (combination_df['Product 2']=='None').sum())
print("Product 3 None:", (combination_df['Product 3']=='None').sum())



Shape: (1656, 9)
Product 2 None: 288
Product 3 None: 720


In [94]:
combination_df.to_csv('combinations_final_mainf2.csv',index = False)

In [96]:

products = ['A','B','C','D']

grid = []

# -----------------------------
# LEVEL 1
# -----------------------------
for p in products:
    grid.append((p,'None','None'))

# -----------------------------
# LEVEL 2
# -----------------------------
for p1 in products:
    for p2 in products:
        if p1 < p2:  # enforce order
            grid.append((p1,p2,'None'))

# -----------------------------
# LEVEL 3
# -----------------------------
for p1 in products:
    for p2 in products:
        for p3 in products:
            if len({p1,p2,p3}) == 3 and sorted([p1,p2,p3]) == [p1,p2,p3]:
                grid.append((p1,p2,p3))

# -----------------------------
# ADD TOTAL
# -----------------------------
grid.append(('All','All','All'))

grid_df = pd.DataFrame(grid, columns=['Product 1','Product 2','Product 3'])

print("Grid size:", grid_df.shape)



Grid size: (15, 3)


In [97]:
results = []

for (date, window), group in user_matrix_df.groupby(['Date','Window']):
    
    for _, row in grid_df.iterrows():
        
        p1,p2,p3 = row
        
        if p1 == 'All':
            users = group['customer_id'].nunique()
        
        else:
            active = [x for x in [p1,p2,p3] if x != 'None']
            
            users = group[
                (group[active].sum(axis=1) == len(active))
            ]['customer_id'].nunique()
        
        results.append({
            'Date': date,
            'Window': window,
            'Join Type': 'AND',
            'Product 1': p1,
            'Product 2': p2,
            'Product 3': p3,
            'Users': users
        })

In [98]:
print("Grid rows:", len(grid_df))
print(grid_df.head())

Grid rows: 15
  Product 1 Product 2 Product 3
0         A      None      None
1         B      None      None
2         C      None      None
3         D      None      None
4         A         B      None


In [99]:
combination_df = pd.DataFrame(results)

In [100]:
print("Final Shape:", combination_df.shape)

print("Product 2 None count:",
      (combination_df['Product 2'] == 'None').sum())

print("Product 3 None count:",
      (combination_df['Product 3'] == 'None').sum())

Final Shape: (540, 7)
Product 2 None count: 144
Product 3 None count: 360


In [101]:
from itertools import combinations
from pathlib import Path

In [ ]:
# to create Combination csv
DIM_PRODUCTS = r"D:\Personal\Data Science\Data Science\End to End Project\SQL Dump\dim_products.csv"
FACT_ORDER_ITEMS = r"D:\Personal\Data Science\Data Science\End to End Project\SQL Dump\fact_order_items.csv"
FACT_ORDERS = r"D:\Personal\Data Science\Data Science\End to End Project\SQL Dump\fact_orders.csv"
OUTPUT_FILE = 'category_combinations_reference_output_v3.csv'


def month_end(dt):
    return pd.Timestamp(dt).to_period('M').to_timestamp('M')


def normalize_categories(values):
    return sorted({str(v).strip() for v in values if pd.notna(v) and str(v).strip()})


def load_base():
    products = pd.read_csv(DIM_PRODUCTS, usecols=['product_id', 'category'])
    items = pd.read_csv(FACT_ORDER_ITEMS, usecols=['order_id', 'product_id'])
    orders = pd.read_csv(FACT_ORDERS, usecols=['order_id', 'customer_id', 'order_date'])
    orders['order_date'] = pd.to_datetime(orders['order_date'])

    base = items.merge(products, on='product_id', how='left').merge(orders, on='order_id', how='left')
    base = base.dropna(subset=['customer_id', 'order_date', 'category']).copy()
    base['category'] = base['category'].astype(str).str.strip()
    base['order_month_end'] = base['order_date'].map(month_end)
    return base


def customer_categories_in_window(base, start_date, end_date):
    tmp = base[(base['order_date'] >= start_date) & (base['order_date'] <= end_date)].copy()
    grouped = tmp.groupby('customer_id')['category'].apply(normalize_categories).reset_index()
    grouped = grouped[grouped['category'].map(len) > 0]
    return grouped


def ordered_states(categories):
    cats = sorted(categories)
    states = set()
    states.add(('and', 'All', 'All', 'All'))

    for c in cats:
        states.add(('and', c, None, None))
        states.add(('and', c, 'All', 'All'))

    for a, b in combinations(cats, 2):
        states.add(('or', a, b, 'All'))
        states.add(('and', a, b, 'All'))

    for a, b, c in combinations(cats, 3):
        states.add(('or', a, b, c))
        states.add(('and', a, b, c))

    def sort_key(x):
        jt, p1, p2, p3 = x
        p1s = '' if p1 is None else str(p1)
        p2s = 'ZZZ' if p2 is None else str(p2)
        p3s = 'ZZZ' if p3 is None else str(p3)
        return (0 if jt == 'and' else 1, p1s, p2s, p3s)

    return sorted(states, key=sort_key)


def actual_counts(customer_sets, current_date, date_value, previous_date, window):
    rows = []
    for _, row in customer_sets.iterrows():
        cats = row['category']
        cust = row['customer_id']
        rows.append((current_date, date_value, 'and', previous_date, 'All', 'All', 'All', window, cust))
        if len(cats) == 1:
            rows.append((current_date, date_value, 'and', previous_date, cats[0], None, None, window, cust))
        for c in cats:
            rows.append((current_date, date_value, 'and', previous_date, c, 'All', 'All', window, cust))
        for a, b in combinations(cats, 2):
            rows.append((current_date, date_value, 'or', previous_date, a, b, 'All', window, cust))
            if len(cats) == 2:
                rows.append((current_date, date_value, 'and', previous_date, a, b, 'All', window, cust))
        for a, b, c in combinations(cats, 3):
            rows.append((current_date, date_value, 'or', previous_date, a, b, c, window, cust))
            if len(cats) == 3:
                rows.append((current_date, date_value, 'and', previous_date, a, b, c, window, cust))

    if not rows:
        return pd.DataFrame(columns=['Current Date','Date','Join Type','Previous Date','Product 1','Product 2','Product 3','Window','Users'])

    out = pd.DataFrame(rows, columns=['Current Date','Date','Join Type','Previous Date','Product 1','Product 2','Product 3','Window','customer_id'])
    out = out.groupby(['Current Date','Date','Join Type','Previous Date','Product 1','Product 2','Product 3','Window'], dropna=False)['customer_id'].nunique().reset_index(name='Users')
    return out


def build_output(base):
    categories = sorted(base['category'].dropna().unique().tolist())
    state_grid = ordered_states(categories)

    max_month = base['order_month_end'].max()
    prev_month = max_month - pd.offsets.MonthEnd(1)
    all_months = sorted(base['order_month_end'].dropna().unique())
    final_rows = []

    for window in ['M', 'Q', 'Y']:
        for date_value in all_months:
            if window == 'M':
                start_date = pd.Timestamp(date_value).to_period('M').start_time.normalize()
            elif window == 'Q':
                start_date = (pd.Timestamp(date_value) - pd.DateOffset(months=2)).to_period('M').start_time.normalize()
            else:
                start_date = (pd.Timestamp(date_value) - pd.DateOffset(months=11)).to_period('M').start_time.normalize()
            end_date = pd.Timestamp(date_value)

            cust_sets = customer_categories_in_window(base, start_date, end_date)
            counts = actual_counts(cust_sets, max_month, date_value, prev_month, window)

            grid = pd.DataFrame(state_grid, columns=['Join Type','Product 1','Product 2','Product 3'])
            grid['Current Date'] = max_month
            grid['Date'] = date_value
            grid['Previous Date'] = prev_month
            grid['Window'] = window

            merged = grid.merge(counts, on=['Current Date','Date','Join Type','Previous Date','Product 1','Product 2','Product 3','Window'], how='left')
            merged['Users'] = merged['Users'].fillna(0).astype(int)
            final_rows.append(merged)

    out = pd.concat(final_rows, ignore_index=True)
    out = out[['Current Date','Date','Join Type','Previous Date','Product 1','Product 2','Product 3','Window','Users']]
    out = out.sort_values(['Window','Date','Join Type','Product 1','Product 2','Product 3'], na_position='last').reset_index(drop=True)
    return out


base = load_base()
output = build_output(base)
output.to_csv(OUTPUT_FILE, index=False)

print('Output created:', OUTPUT_FILE)
print('Rows:', len(output))
print('States per window:')
print(output[['Join Type','Product 1','Product 2','Product 3','Window']].drop_duplicates().groupby('Window').size())
print('Rows where Product 1 = All:')
print(output[output['Product 1'].eq('All')].groupby('Window').size())
output.head(20)

Output created: category_combinations_reference_output_v3.csv
Rows: 1044
States per window:
Window
M    29
Q    29
Y    29
dtype: int64
Rows where Product 1 = All:
Window
M    12
Q    12
Y    12
dtype: int64


,Current Date,Date,Join Type,Previous Date,Product 1,Product 2,Product 3,Window,Users
0,2024-12-31,2024-01-31,and,2024-11-30,All,All,All,M,30
1,2024-12-31,2024-01-31,and,2024-11-30,Beauty,All,All,M,9
2,2024-12-31,2024-01-31,and,2024-11-30,Beauty,Electronics,All,M,1
3,2024-12-31,2024-01-31,and,2024-11-30,Beauty,Electronics,Fashion,M,1
4,2024-12-31,2024-01-31,and,2024-11-30,Beauty,Electronics,Home & Kitchen,M,1
5,2024-12-31,2024-01-31,and,2024-11-30,Beauty,Fashion,All,M,1
6,2024-12-31,2024-01-31,and,2024-11-30,Beauty,Fashion,Home & Kitchen,M,0
7,2024-12-31,2024-01-31,and,2024-11-30,Beauty,Home & Kitchen,All,M,1
8,2024-12-31,2024-01-31,and,2024-11-30,Beauty,None,None,M,4
9,2024-12-31,2024-01-31,and,2024-11-30,Electronics,All,All,M,20


In [ ]:
# to create Num_products CSV

DIM_PRODUCTS = r"D:\Personal\Data Science\Data Science\End to End Project\SQL Dump\dim_products.csv"
FACT_ORDER_ITEMS = r"D:\Personal\Data Science\Data Science\End to End Project\SQL Dump\fact_order_items.csv"
FACT_ORDERS = r"D:\Personal\Data Science\Data Science\End to End Project\SQL Dump\fact_orders.csv"
OUTPUT_FILE = 'num_products_reference_output.csv'


def month_end(dt):
    return pd.Timestamp(dt).to_period('M').to_timestamp('M')


def load_base():
    orders = pd.read_csv(FACT_ORDERS, usecols=['order_id', 'customer_id', 'order_date'])
    items = pd.read_csv(FACT_ORDER_ITEMS, usecols=['order_id', 'product_id'])
    prod = pd.read_csv(DIM_PRODUCTS, usecols=['product_id', 'category'])

    orders['order_date'] = pd.to_datetime(orders['order_date'])
    base = items.merge(prod, on='product_id', how='left').merge(orders, on='order_id', how='left')
    base = base.dropna(subset=['customer_id', 'order_date', 'category']).copy()
    base['order_month_end'] = base['order_date'].map(month_end)
    base['category'] = base['category'].astype(str).str.strip()
    return base


def customer_counts_in_window(base, start_date, end_date):
    tmp = base[(base['order_date'] >= start_date) & (base['order_date'] <= end_date)].copy()
    if tmp.empty:
        return pd.DataFrame(columns=['customer_id', 'num_products'])
    grouped = tmp.groupby('customer_id')['category'].nunique().reset_index(name='num_products')
    grouped = grouped[grouped['num_products'] > 0]
    grouped.loc[grouped['num_products'] >= 3, 'num_products'] = 3
    return grouped


def build_output(base):
    max_month = base['order_month_end'].max()
    prev_month = max_month - pd.offsets.MonthEnd(1)
    all_months = sorted(base['order_month_end'].dropna().unique())

    rows = []
    for window in ['M', 'Q', 'Y']:
        for date_value in all_months:
            if window == 'M':
                start_date = pd.Timestamp(date_value).to_period('M').start_time.normalize()
            elif window == 'Q':
                start_date = (pd.Timestamp(date_value) - pd.DateOffset(months=2)).to_period('M').start_time.normalize()
            else:
                start_date = (pd.Timestamp(date_value) - pd.DateOffset(months=11)).to_period('M').start_time.normalize()
            end_date = pd.Timestamp(date_value)

            counts = customer_counts_in_window(base, start_date, end_date)
            agg = counts.groupby('num_products')['customer_id'].nunique().reset_index(name='Users')

            for num in [1, 2, 3]:
                users = int(agg.loc[agg['num_products'].eq(num), 'Users'].sum())
                rows.append([max_month, date_value, num, prev_month, window, users])

    out = pd.DataFrame(rows, columns=['Current Date', 'Date', 'Num Products', 'Previous Date', 'Window', 'Users'])
    out = out.sort_values(['Window', 'Date', 'Num Products']).reset_index(drop=True)
    return out


def main():
    base = load_base()
    out = build_output(base)
    out.to_csv(OUTPUT_FILE, index=False)
    print('Output created:', OUTPUT_FILE)
    print('Rows:', len(out))
    print(out.head(15).to_string(index=False))


if __name__ == '__main__':
    main()

Output created: num_products_reference_output.csv
Rows: 108
Current Date       Date  Num Products Previous Date Window  Users
  2024-12-31 2024-01-31             1    2024-11-30      M     18
  2024-12-31 2024-01-31             2    2024-11-30      M     10
  2024-12-31 2024-01-31             3    2024-11-30      M      2
  2024-12-31 2024-02-29             1    2024-11-30      M     36
  2024-12-31 2024-02-29             2    2024-11-30      M     15
  2024-12-31 2024-02-29             3    2024-11-30      M      7
  2024-12-31 2024-03-31             1    2024-11-30      M     62
  2024-12-31 2024-03-31             2    2024-11-30      M     35
  2024-12-31 2024-03-31             3    2024-11-30      M     22
  2024-12-31 2024-04-30             1    2024-11-30      M     77
  2024-12-31 2024-04-30             2    2024-11-30      M     48
  2024-12-31 2024-04-30             3    2024-11-30      M     21
  2024-12-31 2024-05-31             1    2024-11-30      M    112
  2024-12-31 202